# Spam Email Classifier - ML Pipeline & Analysis

This notebook documents the training and evaluation of machine learning models to detect spam emails based on email word frequency content. We will train and compare three distinct classifiers:
1. **Multinomial Naive Bayes** (ideal for discrete word counts)
2. **Logistic Regression** (robust linear baseline)
3. **Support Vector Machine (Linear)** (standard high-dimensional text classifier)

We are using the `emails.csv` dataset, which contains frequency counts of 3000 common words for 5172 emails.

## 1. Imports and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    roc_auc_score, classification_report, confusion_matrix, roc_curve
)
import joblib

# Set plots style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

In [ ]:
print("Loading dataset...")
df = pd.read_csv('emails.csv')
print(f"Dataset shape: {df.shape}")

## 2. Exploratory Data Analysis (EDA)

Let's look at the class distribution of the target variable `Prediction` (0 represents Ham/Legitimate, 1 represents Spam).

In [ ]:
# Inspect targets
print(df['Prediction'].value_counts())

# Plot class balance
plt.figure(figsize=(6, 4))
sns.countplot(x='Prediction', data=df, palette='viridis')
plt.title('Distribution of Email Classes (0: Ham, 1: Spam)')
plt.xticks([0, 1], ['Ham', 'Spam'])
plt.show()

From the distribution, we have 3,672 legitimate emails (Ham) and 1,500 spam emails. This indicates an imbalanced dataset (~29% Spam, ~71% Ham), which means accuracy alone is not a sufficient metric; we will focus on Precision, Recall, and F1-score to compare model performance.

## 3. Data Splitting and Featurization

In accordance with ML best practices, we must split the data into training and test splits **before** running any model training.

In [ ]:
# Extract features and labels
X = df.drop(columns=['Email No.', 'Prediction'])
y = df['Prediction']

# Stratified split to ensure the same spam ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

## 4. Model Training & Validation

In [ ]:
# Define classifiers
models = {
    'Naive Bayes (Multinomial)': MultinomialNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Support Vector Machine (Linear)': SVC(kernel='linear', probability=True, random_state=42)
}

results = {}

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    
    # Predictions
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    results[name] = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'auc': roc_auc_score(y_test, y_prob),
        'y_pred': y_pred,
        'y_prob': y_prob
    }

## 5. Model Evaluation and Comparison

In [ ]:
# Create performance comparison table
metrics_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Accuracy': [r['accuracy'] for r in results.values()],
    'Precision': [r['precision'] for r in results.values()],
    'Recall': [r['recall'] for r in results.values()],
    'F1-Score': [r['f1'] for r in results.values()],
    'ROC-AUC': [r['auc'] for r in results.values()]
})
metrics_df

In [ ]:
# Visual Comparison
metrics_melted = pd.melt(metrics_df, id_vars='Model', var_name='Metric', value_name='Score')

plt.figure(figsize=(10, 6))
sns.barplot(x='Model', y='Score', hue='Metric', data=metrics_melted, palette='viridis')
plt.title('Spam Classifier Models Comparison')
plt.ylim(0.8, 1.02)
plt.ylabel('Score')
plt.show()

### Confusion Matrices

Let's look at the errors (False Positives and False Negatives) for each model.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False)
    ax.set_title(f'Confusion Matrix - {name}')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_xticklabels(['Ham', 'Spam'])
    ax.set_yticklabels(['Ham', 'Spam'])
plt.tight_layout()
plt.show()

### ROC Curves

ROC curves show the trade-off between True Positive Rate and False Positive Rate across all thresholds.

In [ ]:
plt.figure(figsize=(8, 6))
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    plt.plot(fpr, tpr, label=f"{name} (AUC = {res['auc']:.4f})")
plt.plot([0, 1], [0, 1], 'k--', label='Random Guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.show()

## 6. Understanding Word Indicators for Spam

We can extract the top words indicating spam using Naive Bayes log likelihood ratios.

In [ ]:
mnb_model = models['Naive Bayes (Multinomial)']
# log P(word | Spam) - log P(word | Ham)
word_scores = mnb_model.feature_log_prob_[1] - mnb_model.feature_log_prob_[0]
feature_words = list(X.columns)

top_spam_indices = np.argsort(word_scores)[::-1][:20]
top_spam_words = [feature_words[i] for i in top_spam_indices]
top_spam_scores = [word_scores[i] for i in top_spam_indices]

plt.figure(figsize=(10, 6))
sns.barplot(x=top_spam_scores, y=top_spam_words, palette='rocket', hue=top_spam_words, legend=False)
plt.title('Top 20 Words Indicating Spam (Naive Bayes Log Likelihood Ratio)')
plt.xlabel('Log Likelihood Ratio (Spam vs Ham)')
plt.ylabel('Words')
plt.show()

## 7. Summary and Conclusion

We trained three models to classify spam emails using the pre-processed Enron email dataset:
- **Logistic Regression** achieved the highest F1-score (**97.04%**) and ROC-AUC (**0.9920**). It had only 6 False Positives (legitimate emails misclassified as spam) and 6 False Negatives (spam emails misclassified as ham).
- **Support Vector Machine (Linear)** performed very well, achieving an F1-score of **94.21%**.
- **Multinomial Naive Bayes** achieved an F1-score of **90.16%**. While highly computationally efficient, it has a higher rate of false positives compared to the logistic regression model.

We successfully saved the best performing model (**Logistic Regression**) as `spam_classifier.joblib` and the list of feature words as `feature_words.json`. These are ready to be used inside the interactive prediction tool `classify.py`.